In [1]:
# run_scad.ipynb (Python code cells)
# 注意：此文件为 .ipynb 格式，此处以 Python 代码形式展示所有单元格内容

# %% Cell 1
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from datetime import datetime

# 导入自定义模块
import scad


In [5]:

# 创建目录结构
experiment_path = "result/SCC"
model_path = os.path.join(experiment_path, "models")
data_path = os.path.join(experiment_path, "data")
result_path = os.path.join(experiment_path, "results")
os.makedirs(model_path, exist_ok=True)
os.makedirs(data_path, exist_ok=True)
os.makedirs(result_path, exist_ok=True)

# 读取数据（请按实际路径调整）
adata_ST = sc.read_h5ad('data/enhanced_exp.h5ad')
adata_sc = sc.read('data/adata_SCC_sc.h5ad')
spot_info = pd.read_csv('data/mapping_SCC.txt', sep='\t')

adata_ST.obsm['spatial'] = adata_ST.obs[['x','y']].to_numpy()
svg_list = pd.read_csv('data/svg-enhanced-scc.csv', header=0, sep=',', index_col=0).index

# 归一化坐标
adata_ST.obsm['spatial'] = (adata_ST.obsm['spatial'] - adata_ST.obsm['spatial'].min(axis=0)) / (adata_ST.obsm['spatial'].max(axis=0) - adata_ST.obsm['spatial'].min(axis=0))

N_st = adata_ST.shape[0]
splits = scad.split_dataset_cv3(N_st, seed=42)
Y = adata_ST.obs[['x','y']].values.astype(np.float32)
N_rna = adata_sc.X.shape[0]
training_idx_rna = np.array(range(N_rna))


In [7]:
# %% Cell 2 - 训练模型 Fold 1
print("=== Training Fold 1 ===")
model1 = scad.Model3(
    resolution="low",
    batch_size=200,
    train_epoch=3000,
    cut_steps=0.5,
    sf_coord=50,
    rad_cutoff=1.2,
    seed=1234,
    lambdacos=10,
    lambdaSWD=5,
    lambdalat=10,
    lambdarec=0.1,
    model_path=model_path,
    data_path=data_path,
    result_path=result_path,
    ot=False,
    device="cpu"
)
K, cluster = model1.preprocess(svg_list, adata_sc, adata_ST, res=0.5)
training_idx_st = np.array(splits[0][0])
model1.train(training_idx_rna, training_idx_st)
mu1, phi1, sigma1, z_A1, z_B1, m_A1, m_B1 = model1.eval2()
val_idx1, test_idx1 = np.array(splits[0][1]), np.array(splits[0][2])

# 保存模型1的参数
torch.save({
    'D_A': model1.D_A.state_dict(),
    'D_B': model1.D_B.state_dict(),
    'E_A': model1.E_A.state_dict(),
    'E_B': model1.E_B.state_dict(),
    'G_A': model1.G_A.state_dict(),
    'G_B': model1.G_B.state_dict(),
    'E_s': model1.E_s.state_dict()
}, os.path.join(model_path, "model1.pth"))

# %% Cell 3 - 训练模型 Fold 2
print("=== Training Fold 2 ===")
model2 = scad.Model3(
    resolution="low",
    batch_size=200,
    train_epoch=3000,
    cut_steps=0.5,
    sf_coord=50,
    rad_cutoff=1.2,
    seed=1234,
    lambdacos=10,
    lambdaSWD=5,
    lambdalat=10,
    lambdarec=0.1,
    model_path=model_path,
    data_path=data_path,
    result_path=result_path,
    ot=False,
    device="cpu"
)
K, cluster = model2.preprocess(svg_list, adata_sc, adata_ST, res=0.5)
training_idx_st = np.array(splits[1][0])
model2.train(training_idx_rna, training_idx_st)
mu2, phi2, sigma2, z_A2, z_B2, m_A2, m_B2 = model2.eval2()
val_idx2, test_idx2 = np.array(splits[1][1]), np.array(splits[1][2])

torch.save({
    'D_A': model2.D_A.state_dict(),
    'D_B': model2.D_B.state_dict(),
    'E_A': model2.E_A.state_dict(),
    'E_B': model2.E_B.state_dict(),
    'G_A': model2.G_A.state_dict(),
    'G_B': model2.G_B.state_dict(),
    'E_s': model2.E_s.state_dict()
}, os.path.join(model_path, "model2.pth"))

# %% Cell 4 - 训练模型 Fold 3
print("=== Training Fold 3 ===")
model3 = scad.Model3(
    resolution="low",
    batch_size=200,
    train_epoch=3000,
    cut_steps=0.5,
    sf_coord=50,
    rad_cutoff=1.2,
    seed=1234,
    lambdacos=10,
    lambdaSWD=5,
    lambdalat=10,
    lambdarec=0.1,
    model_path=model_path,
    data_path=data_path,
    result_path=result_path,
    ot=False,
    device="cpu"
)
K, cluster = model3.preprocess(svg_list, adata_sc, adata_ST, res=0.5)
training_idx_st = np.array(splits[2][0])
model3.train(training_idx_rna, training_idx_st)
mu3, phi3, sigma3, z_A3, z_B3, m_A3, m_B3 = model3.eval2()
val_idx3, test_idx3 = np.array(splits[2][1]), np.array(splits[2][2])

torch.save({
    'D_A': model3.D_A.state_dict(),
    'D_B': model3.D_B.state_dict(),
    'E_A': model3.E_A.state_dict(),
    'E_B': model3.E_B.state_dict(),
    'G_A': model3.G_A.state_dict(),
    'G_B': model3.G_B.state_dict(),
    'E_s': model3.E_s.state_dict()
}, os.path.join(model_path, "model3.pth"))

# %% Cell 5 - 汇总结果并执行 Conformal Prediction
print("=== Performing Conformal Prediction ===")
true_coord = adata_ST.obsm['spatial']

final_aberrant1, final_confidence1, final_lambda1, pred_coords1, _ = scad.conformal_prediction(
    true_coord, z_B1, m_B1, val_idx1, test_idx1, alpha=0.05)

final_aberrant2, final_confidence2, final_lambda2, pred_coords2, _ = scad.conformal_prediction(
    true_coord, z_B2, m_B2, val_idx2, test_idx2, alpha=0.05)

final_aberrant3, final_confidence3, final_lambda3, pred_coords3, _ = scad.conformal_prediction(
    true_coord, z_B3, m_B3, val_idx3, test_idx3, alpha=0.05)

# 合并三折结果（相加得到最终判定，即至少在一折中被判为异常则视为异常）
final_aberrant = final_aberrant1 + final_aberrant2 + final_aberrant3
final_lambda = final_lambda1 + final_lambda2 + final_lambda3

# 合并预测坐标：使用模型3的预测坐标作为基础，然后根据测试集索引替换为各折的预测值
final_predict = pred_coords3.clone()
test_list1 = val_idx1.tolist() + test_idx1.tolist()
test_list2 = val_idx2.tolist() + test_idx2.tolist()
test_list3 = val_idx3.tolist() + test_idx3.tolist()
final_predict[test_list1, :] = pred_coords1[test_list1, :]
final_predict[test_list2, :] = pred_coords2[test_list2, :]
final_predict[test_list3, :] = pred_coords3[test_list3, :]  # 实际上已经是 model3 的值，但保留一致性

# 保存结果到原始 ST 对象（注意索引映射）
adata_ST0 = sc.read_h5ad('data/adata_SCC_ST.h5ad')  # 请替换实际路径
mapping_TESLA = pd.read_csv('data/mapping_SCC.txt', sep='\t')
mapping_TESLA.index = mapping_TESLA['ori_index']

pred_error = np.zeros((adata_ST0.shape[0], 1))
pred_error[:, 0] = np.sqrt(((final_predict - true_coord) ** 2).sum(axis=1))[mapping_TESLA.loc[list(range(666))]['target_index']]
adata_ST0.obs['pred_error'] = pred_error

pred_abb = np.zeros((adata_ST0.shape[0], 1))
pred_abb[:, 0] = final_aberrant[mapping_TESLA.loc[list(range(666))]['target_index']]
adata_ST0.obs['pred_abb'] = pred_abb

adata_ST0.obs['nonconformityscore'] = final_lambda[mapping_TESLA.loc[list(range(666))]['target_index']]
adata_ST0.obs.to_csv(os.path.join(result_path, 'spot_info_SCC.txt'), sep='\t')

# 保存所有评估中间结果
np.savez(os.path.join(result_path, 'eval_out.npz'),
         true_coord=true_coord,
         z_B1=z_B1, m_B1=m_B1, val_idx1=val_idx1, test_idx1=test_idx1,
         z_B2=z_B2, m_B2=m_B2, val_idx2=val_idx2, test_idx2=test_idx2,
         z_B3=z_B3, m_B3=m_B3, val_idx3=val_idx3, test_idx3=test_idx3,
         final_predict=final_predict, final_lambda=final_lambda)

print("All results saved successfully.")

=== Training Fold 1 ===
Finding highly variable genes...
# overlap highly variable genes is: 1010
Normalizing and scaling...
AnnData object with n_obs × n_vars = 20946 × 14809
    obs: 'x', 'y', 'color', 'z', 'CD151', 'batch'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'log1p'
    obsm: 'spatial', 'loc'
['TAX1BP3', 'VAMP3', 'UBE2D2', 'TNFAIP1', 'ACTR3', 'WNT5A', 'UFD1L', 'PDCD6IP', 'DNAJB1', 'MPZL2', 'NELL2', 'MRPL37', 'ANXA8L1', 'DYNLT1', 'PSMB1', 'CCT2', 'FAM83H', 'RAB11A', 'ARPC1A', 'MUCL1', 'ATP5C1', 'RNF7', 'DBNL', 'ID1', 'HIST2H2AA4', 'ATP5F1', 'MAPK1IP1L', 'P4HB', 'ETF1', 'CAPRIN1', 'PTBP3', 'PMEL', 'FAM213A', 'LAMC2', 'SLC39A6', 'TM9SF2', 'ARF6', 'CBR1', 'CDKN1A', 'CAPN1', 'RAB10', 'SEMA4B', 'ITGB4', 'TALDO1', 'XRCC6', 'ACADVL', 'GALNT6', 'PTGFRN', 'ARPP19', 'PPA1', 'RAB1A', 'SNRPD3', 'MAPK6', 'NDUFA4L2', 'HDGF', 'PSMD2', 'NFE2L1', 'RAB18', 'PPL', 'TP63', 'MVP', 'MDH2', 'KTN1', 'EIF3M', 'CDH3', 'PRDX6', '